# Bài 8
Đây là notebook chứa mã nguồn đầy đủ của bài 8.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
def solve_bai08(discount=0.05, capital_growth=0.06, target_ai=0.85, budget_growth=0.08):
    T = 10
    years = list(range(2026, 2036))
    
    def simulate(strategy='opt', shock_year=None):
        K = [25900.0]
        Y = []
        C = []
        D = [100.0]
        H = [500.0]
        AI = [0.60]
        
        welfare = 0.0
        
        for t in range(T):
            A = 5.0
            if shock_year == years[t]:
                A *= 0.92  # 8% drop in Y
                
            y_t = A * (K[t]**0.4) * (H[t]**0.3) * (D[t]**0.1) * (AI[t]**0.2)
            Y.append(y_t)
            
            if strategy == 'even':
                inv_rate = 0.25
                d_rate = 0.05
                h_rate = 0.05
            elif strategy == 'front':
                inv_rate = 0.35 if t < 5 else 0.15
                d_rate = 0.08 if t < 5 else 0.02
                h_rate = 0.08 if t < 5 else 0.02
            else: # 'opt'
                inv_rate = 0.28
                d_rate = 0.06
                h_rate = 0.06
                
            c_t = y_t * (1.0 - inv_rate - d_rate - h_rate)
            C.append(c_t)
            
            welfare += (c_t ** 0.5) / ((1 + discount) ** t)
            
            if t < T - 1:
                K.append(K[t] * (1 - 0.05) + y_t * inv_rate)
                D.append(D[t] * (1 - 0.10) + y_t * d_rate)
                H.append(H[t] * (1 - 0.02) + y_t * h_rate)
                ai_next = AI[t] + 0.05 * (D[t]/100.0)
                AI.append(min(1.0, ai_next))
                
        return Y, K, C, D, H, AI, welfare

    Y_opt, K_opt, C_opt, D_opt, H_opt, AI_opt, w_opt = simulate('opt')
    Y_shock, _, _, _, _, _, w_shock = simulate('opt', shock_year=2028)
    _, _, _, _, _, _, w_even = simulate('even')
    _, _, _, _, _, _, w_front = simulate('front')
    
    better_strat = "Front-load (Đầu tư mạnh giai đoạn đầu)" if w_front > w_even else "Trải đều (Đầu tư ổn định)"

    return {
        'years': years,
        'welfare_opt': float(w_opt),
        'K': [float(x) for x in K_opt],
        'Y': [float(x) for x in Y_opt],
        'C': [float(x) for x in C_opt],
        'D': [float(x) for x in D_opt],
        'H': [float(x) for x in H_opt],
        'AI': [float(x) for x in AI_opt],
        'welfare_shock': float(w_shock),
        'Y_shock': [float(x) for x in Y_shock],
        'welfare_even': float(w_even),
        'welfare_front': float(w_front),
        'better_strategy': better_strat
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai08()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())